# RPS Classification - 지난주 기본 실습

가위바위보 이미지를 DenseNet121 전이학습으로 분류하는 기본 실습입니다.
이번 QAT 실습 노트북과 비교하기 위해 같은 `RPS_Dataset`을 사용합니다.


## 모듈 로딩

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob
import cv2
import shutil
import subprocess
from pathlib import Path
from sklearn.model_selection import train_test_split

print("NumPy Version :{}".format(np.__version__))
print("TensorFlow Version :{}".format(tf.__version__))
print("Matplotlib Version :{}".format(plt.matplotlib.__version__))


## 실행 환경 확인


In [ ]:
# Colab인지, GCP/VSCode 같은 일반 Python 환경인지 확인합니다.
# 데이터셋은 다음 셀에서 저장소 내부의 RPS_Dataset 폴더를 자동으로 찾습니다.
try:
    import google.colab  # type: ignore
    colab = True
    print("Colab environment.")
except Exception:
    colab = False
    print("Local/GCP/VSCode environment.")


## RPS 데이터셋 준비

학생용 저장소는 `RPS_Dataset` 폴더를 저장소 안에 포함합니다.
따라서 Google Drive나 zip 압축 해제 없이 Colab, GCP GPU 서버, VSCode에서 같은 방식으로 실행할 수 있습니다.


In [ ]:
# 학생용 저장소 구조
# RP2-RPS-QAT-Lab/
#   RPS_Dataset/
#   notebooks/
#     RPS_Classification_DenseNet121_지난주실습.ipynb
#
# 학생이 이 저장소를 fork/clone했다면 별도 다운로드 없이 RPS_Dataset을 바로 사용합니다.

GITHUB_REPO_URL = "https://github.com/philipdekim-OnD01/RP2-RPS-QAT-Lab.git"
COLAB_REPO_DIR = Path("/content/RP2-RPS-QAT-Lab")
SERVER_REPO_DIR = Path.home() / "RP2-RPS-QAT-Lab_data"

def get_git_root():
    """현재 노트북이 Git 저장소 안에서 실행 중이면 저장소 루트 경로를 반환합니다."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            check=True, capture_output=True, text=True
        )
        return Path(result.stdout.strip())
    except Exception:
        return None

def clone_repo_if_needed(target_dir):
    """데이터셋이 없을 때 학생용 GitHub 저장소를 clone합니다."""
    if not target_dir.exists():
        print("Clone GitHub repository for RPS_Dataset:", GITHUB_REPO_URL)
        subprocess.run(
            ["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(target_dir)],
            check=True
        )
    return target_dir

cwd = Path.cwd()
git_root = get_git_root()

candidate_paths = []
if git_root is not None:
    candidate_paths.append(git_root / "RPS_Dataset")

candidate_paths.extend([
    cwd / "RPS_Dataset",
    cwd / "../RPS_Dataset",
    Path("RPS_Dataset"),
    Path("../RPS_Dataset"),
])

files_path = None
for candidate in candidate_paths:
    if candidate.exists():
        files_path = str(candidate.resolve())
        break

if files_path is None:
    clone_dir = COLAB_REPO_DIR if colab else SERVER_REPO_DIR
    repo_dir = clone_repo_if_needed(clone_dir)
    dataset_dir = repo_dir / "RPS_Dataset"
    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"RPS_Dataset 폴더를 찾을 수 없습니다: {dataset_dir}. "
            "GITHUB_REPO_URL이 올바른지 확인하세요."
        )
    files_path = str(dataset_dir.resolve())

print("Working directory:", cwd)
print("Git root:", git_root)
print("RPS dataset path:", files_path)
print("Class folders:", sorted([p.name for p in Path(files_path).iterdir() if p.is_dir()]))


## 폴더 별로 파일 읽어 데이터화 진행

In [ ]:
%%time
first = True
IMG_SIZE = 64
for ind in range(0, 3, 1) :
    path = files_path + '/' + str(ind) + '/*.*'
    print(path)
    files = sorted(glob.glob(path))
    if len(files) == 0:
        raise FileNotFoundError(f"No image files found: {path}")
    tmpx = np.array([(cv2.resize(cv2.cvtColor(cv2.imread(x, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB), (IMG_SIZE,IMG_SIZE))) for x in files])
    tmpy = np.array([ind] * len(files))

    xtrain, xtest, ytrain, ytest = train_test_split(tmpx, tmpy, test_size=0.2, random_state=123)

    if first == True:
        train_data = xtrain.copy()
        train_labels = ytrain.copy()
        test_data = xtest.copy()
        test_labels = ytest.copy()
        first = False
    else :
        train_data = np.concatenate((train_data, xtrain))
        train_labels = np.concatenate((train_labels, ytrain))
        test_data = np.concatenate((test_data, xtest))
        test_labels = np.concatenate((test_labels, ytest))


## 데이터 확인

In [ ]:
# 0~1로 변환 처리 안함 (모델 내에서 처리함)
# train_data = train_data / 255.0
# test_data = test_data / 255.0

print(train_data.shape)
print(train_labels.shape)
print(test_data.shape)
print(test_labels.shape)

## 모델 생성


In [ ]:
denseNet = tf.keras.applications.densenet.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.densenet.preprocess_input(inputs)
x = denseNet(x, training = False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

## 모델 컴파일

In [ ]:
model.compile(optimizer= tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 학습 전 상황

In [ ]:
def Make_Result_Plot(suptitle, data, label, y_max):
    size = data.shape[0]//10
    fig_result, ax_result = plt.subplots(2,5,figsize=(18, 7))
    fig_result.suptitle(suptitle)
    for idx in range(10):
        cnt = idx * size + size//2
        ax_result[idx//5,idx%5].imshow(data[cnt].reshape((IMG_SIZE,IMG_SIZE, 3)))
        ax_result[idx//5,idx%5].set_title("test_data[{}] (label : {} / y : {})".format(cnt, label[cnt], y_max[cnt]))

In [ ]:
y_out = model.predict(test_data)
y_max = np.argmax(y_out, axis=1).reshape((-1, 1))
Make_Result_Plot("Before Training", test_data, test_labels, y_max)

## 콜백 설정

In [ ]:
savedModelName = 'RPS_PreTrained_DenseNet.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(savedModelName,
                                    save_best_only=True)
]

## 모델 학습

In [ ]:
%%time
history = model.fit(train_data, train_labels, epochs=30,
                    callbacks=callbacks,
                    validation_data=(test_data, test_labels))

## Ploting : Cost/Training Count

In [ ]:
plt.figure(figsize=(6, 10))
plt.subplot(2, 1, 1)
plt.title('Accuray')
plt.plot(history.history['accuracy'], 'b', label='train_accuracy')
plt.plot(history.history['val_accuracy'], 'g', label='val_accuracy')
plt.grid(True)
plt.ylabel('Accuracy')
plt.legend(loc='best')

plt.subplot(2, 1, 2)
plt.title('Loss')
plt.plot(history.history['loss'], 'b', label='train_loss')
plt.plot(history.history['val_loss'], 'g', label='val_loss')
plt.grid(True)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='best')
plt.show()

## best 모델 로딩 및 테스트

In [ ]:
model_best = tf.keras.models.load_model(savedModelName)

In [ ]:
y_out = model_best.predict(test_data)
y_max = np.argmax(y_out, axis=1).reshape((-1, 1))
Make_Result_Plot("After Training", test_data, test_labels, y_max)

## best 모델 백업

In [ ]:
# 학습 결과는 저장소 안의 results 폴더에 저장합니다.
# results 폴더는 .gitignore에 포함되어 있어 큰 모델 파일이 실수로 GitHub에 올라가지 않습니다.
repo_root = git_root if git_root is not None else Path.cwd()
save_dir = repo_root / "results"
save_dir.mkdir(parents=True, exist_ok=True)
print("Save directory:", save_dir)


In [ ]:
# copy saved model
saved_model_path = save_dir / savedModelName
shutil.copy2(savedModelName, saved_model_path)
print("Saved model copied to:", saved_model_path)


## LiteRT 모델로 변환

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_best)
tflite_model = converter.convert()

## LiteRT 모델을 파일로 저장

In [ ]:
tfliteFileName = 'RPS_PreTrained_DenseNet.tflite'

In [ ]:
tflite_path = save_dir / tfliteFileName
tflite_path.write_bytes(tflite_model)
